# 04 · Complementariedad con filtrado colaborativo (co-compras)
Fase 4 del plan: dado un artículo ancla, devolver prendas **complementarias** (las que la gente compra junto con él).

**Inputs:** salida de `01_eda` y de `03_embeddings` (*Add Input → Your Work*). No necesita GPU.

**Salida** (la usa el notebook 05 y la app):
- `copurchase.parquet`: top-50 artículos co-comprados por ancla, **sin filtrar** (señal para el ranking).
- `complements.parquet`: top-50 **complementos** por ancla (otro grupo de producto, misma línea), para todos los artículos con imagen.
  Si el ancla no tiene co-compras se usa la estrategia de arranque en frío que gane en validación (columna `fuente`).
- `complementary_eval.csv`: recall@k en la semana de validación.

In [ ]:
import os, glob, time
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp

KAGGLE = os.path.exists("/kaggle/input")
def find(name):
    """Busca un archivo en los Inputs del notebook (Kaggle) o en data/ (local)."""
    hits = glob.glob(f"{'/kaggle/input' if KAGGLE else 'data'}/**/{name}", recursive=True)
    assert hits, f"No encontré {name}: agrega como Input el notebook que lo genera"
    return Path(hits[0])

OUT = Path("/kaggle/working") if KAGGLE else Path("data/04-complementary-output")
OUT.mkdir(parents=True, exist_ok=True)

articles = pd.read_parquet(find("articles.parquet"))
train = pd.read_parquet(find("train_small.parquet"))   # 12 semanas antes de validación
val = pd.read_parquet(find("val.parquet"))
print(train.shape, val.shape)

# Todo se trabaja con el número de fila del artículo en `articles` (enteros: rápido y ligero)
row = pd.Series(np.arange(len(articles)), index=articles.article_id)
AID = articles.article_id.to_numpy()
PG = pd.factorize(articles.product_group_name)[0]   # grupo de producto (Upper body, Lower body, Shoes, ...)
IG = pd.factorize(articles.index_group_name)[0]     # línea (Ladieswear, Menswear, Baby/Children, Divided, Sport)

def is_complement(a, b):
    """b complementa a a: distinto grupo de producto y misma línea (no mezclar ropa de bebé con la de mujer)."""
    return (PG[a] != PG[b]) & (IG[a] == IG[b])

## Canastas
El dataset no tiene sesiones (`t_dat` es solo la fecha), así que **canasta = mismo cliente, mismo día**.
Se descartan canastas de más de 30 artículos (0.1 %, típicamente revendedores) porque generan pares sin relación real.

In [ ]:
MAX_BASKET = 30

def baskets(tx):
    b = tx[["customer_id", "t_dat", "article_id"]].drop_duplicates()
    b["basket"] = b.groupby(["customer_id", "t_dat"]).ngroup()
    b = b[b.groupby("basket").article_id.transform("size") <= MAX_BASKET]
    b["i"] = row.loc[b.article_id].to_numpy()
    return b[["basket", "i"]]

tb = baskets(train)
print("Canastas en train:", tb.basket.nunique(), "| tamaño medio:", round(len(tb) / tb.basket.nunique(), 2))

## Matriz de co-compras
Con la matriz binaria canasta × artículo `B`, la matriz de co-compras es `C = Bᵀ·B`:
`C[i, j]` = número de canastas donde aparecen juntos `i` y `j`; la diagonal = canastas que contienen `i`.

El conteo crudo favorece a los artículos más vendidos (salen con todo), así que también probamos la
**similitud coseno** `C[i,j] / √(n_i · n_j)`, que descuenta la popularidad.

In [ ]:
B = sp.csr_matrix((np.ones(len(tb), np.float32), (pd.factorize(tb.basket)[0], tb.i)),
                  shape=(tb.basket.nunique(), len(articles)))
C = (B.T @ B).tocoo()
n = np.asarray(B.sum(0)).ravel()   # canastas que contienen cada artículo

pairs = pd.DataFrame({"a": C.row, "b": C.col, "count": C.data})
pairs = pairs[pairs.a != pairs.b].reset_index(drop=True)
pairs["cosine"] = (pairs["count"] / np.sqrt(n[pairs.a] * n[pairs.b])).astype(np.float32)
pairs["comp"] = is_complement(pairs.a.to_numpy(), pairs.b.to_numpy())
print(f"{len(pairs):,} pares co-comprados | {pairs.comp.mean():.1%} son complementarios | "
      f"{pairs.a.nunique():,} artículos con co-compras")

In [ ]:
def top_k(df, score, k=50):
    """Para cada ancla `a`, los k mejores `b` según `score`, con su posición (`rank`, desde 0)."""
    t = df.sort_values(["a", score], ascending=[True, False]).groupby("a").head(k)
    return t.assign(rank=t.groupby("a").cumcount())

## Evaluación: recall@k de co-compras reales (semana de validación)
Para cada canasta de validación con ≥2 artículos, tomamos cada artículo como ancla y vemos cuántos de los
**otros** artículos de su canasta aparecen en sus top-k recomendaciones (construidas solo con train → sin fuga).

- **Complementos:** solo cuentan como objetivo los artículos de la canasta que son complementarios del ancla.
- **General:** cuentan todos (incluye otra talla/color de lo mismo); es la señal que usará el ranking del 05.

Referencia: **popularidad**, los k más vendidos de la última semana de train que cumplen el mismo filtro.

In [ ]:
K = 12
vb = baskets(val)
vp = vb.merge(vb, on="basket").rename(columns={"i_x": "a", "i_y": "b"})
vp = vp[vp.a != vp.b]
vp_comp = vp[is_complement(vp.a.to_numpy(), vp.b.to_numpy())]
print(f"Pares en validación: {len(vp):,} generales, {len(vp_comp):,} complementarios")

def recall_at_k(recs, truth, k=K):
    """Recall@k promedio por (canasta, ancla). recs: columnas a, b, rank. truth: basket, a, b."""
    hits = truth.merge(recs.loc[recs["rank"] < k, ["a", "b"]].assign(hit=1), on=["a", "b"], how="left")
    return hits.hit.notna().groupby([hits.basket, hits.a]).mean().mean()

last_week = train[train.t_dat > train.t_dat.max() - pd.Timedelta(days=7)]
sales = last_week.article_id.value_counts()
pop = row.loc[sales.index].to_numpy()                       # artículos ordenados por ventas
POP = np.zeros(len(articles)); POP[pop] = sales / sales.max()  # popularidad en [0, 1]

def popularity_recs(anchors, complement, k=K):
    """Top-k popular para cada ancla (sin el ancla); con complement=True solo artículos complementarios del ancla.
    Solo depende del (grupo, línea) del ancla, así que se calcula una vez por combinación."""
    a = pd.DataFrame({"a": np.unique(anchors)})
    a["key"] = PG[a.a] * 100 + IG[a.a] if complement else 0
    rep = a.drop_duplicates("key")
    per_key = pd.concat([pd.DataFrame({"key": key, "b": (pop[is_complement(x, pop)] if complement else pop)[:k + 1]})
                         for x, key in zip(rep.a, rep.key)])
    r = a.merge(per_key, on="key")
    r = r[r.a != r.b]
    r = r.assign(rank=r.groupby("a").cumcount())
    return r[r["rank"] < k]

warm = set(pairs.a)
pop_comp = popularity_recs(vp_comp.a, True)
results = {"Popularidad": {"complementos": recall_at_k(pop_comp, vp_comp),
                           "general": recall_at_k(popularity_recs(vp.a, False), vp)}}
for score in ["count", "cosine"]:
    results[f"Co-compra ({score})"] = {
        "complementos": recall_at_k(top_k(pairs[pairs.comp], score, K), vp_comp),
        "general": recall_at_k(top_k(pairs, score, K), vp),
    }
res = pd.DataFrame(results).T.add_prefix(f"recall@{K} ")
print(f"Anclas de validación con co-compras en train: {vp.a.isin(warm).mean():.1%}")
res.style.format("{:.4f}")

In [ ]:
BEST = res[f"recall@{K} complementos"].drop("Popularidad").idxmax().split("(")[1][:-1]
print("Score elegido:", BEST)

## Arranque en frío: anclas sin co-compras
Una prenda nueva no tiene historial. Probamos dos estrategias sobre las anclas de validación sin co-compras en train
y nos quedamos con la que tenga mejor recall:
- **Vecino visual:** buscar el artículo visualmente más parecido (embeddings del 03) que sí tenga co-compras y usar
  sus complementos, re-filtrados respecto al ancla original (conserva el estilo).
- **Popularidad:** los más vendidos de la última semana entre los grupos complementarios del ancla.

Nota: la foto que sube el usuario en el chat no pasa por aquí. El 05 la mapea a los artículos disponibles más
parecidos, que casi siempre tienen co-compras.

In [ ]:
emb = np.load(find("img_emb.npy"))
emb_rows = row.loc[pd.read_parquet(find("emb_ids.parquet")).article_id].to_numpy()   # fila de articles de cada embedding
pos = pd.Series(np.arange(len(emb_rows)), index=emb_rows)                             # fila de articles → fila de emb

def knn(Q, E, k, chunk=1024):
    """Top-k de E por producto punto para cada fila de Q → (scores, índices), ordenados de mayor a menor."""
    Q = np.atleast_2d(Q)
    S_all, I_all = [], []
    for s in range(0, len(Q), chunk):
        S = Q[s:s + chunk] @ E.T
        I = np.argpartition(-S, k, axis=1)[:, :k]
        Sk = np.take_along_axis(S, I, 1)
        o = np.argsort(-Sk, axis=1)
        S_all.append(np.take_along_axis(Sk, o, 1)); I_all.append(np.take_along_axis(I, o, 1))
    return np.concatenate(S_all), np.concatenate(I_all)

def ranked(df):
    df = df.sort_values(["a", "score"], ascending=[True, False])
    return df.assign(rank=df.groupby("a").cumcount())

direct = top_k(pairs[pairs.comp], BEST, 50)[["a", "b", BEST]].rename(columns={BEST: "score"}).assign(fuente="co-compra")
has_comp = np.array(sorted(set(direct.a) & set(emb_rows)))
cold = np.setdiff1d(emb_rows, direct.a.unique())
t0 = time.time()
sim, nn = knn(emb[pos.loc[cold]], emb[pos.loc[has_comp]], 1)
proxy = pd.DataFrame({"a": cold, "via": has_comp[nn[:, 0]]})
print(f"{len(cold):,} artículos con imagen sin complementos por co-compra ({time.time() - t0:.0f}s de búsqueda)")

borrowed = proxy.merge(direct.drop(columns="fuente").rename(columns={"a": "via"}), on="via")
borrowed = ranked(borrowed[is_complement(borrowed.a.to_numpy(), borrowed.b.to_numpy()) & (borrowed.a != borrowed.b)]
                  .assign(fuente="vecino visual"))
popular = popularity_recs(cold, True, 50)
popular = popular.assign(score=POP[popular.b], fuente="popularidad")

cold_val = vp_comp[~vp_comp.a.isin(direct.a)]
cold_res = {"vecino visual": recall_at_k(borrowed, cold_val), "popularidad": recall_at_k(pop_comp, cold_val)}
COLD = max(cold_res, key=cold_res.get)
print(f"Anclas frías de validación ({cold_val.groupby(['basket', 'a']).ngroups:,} casos) · recall@{K}:",
      {k: round(v, 4) for k, v in cold_res.items()}, "→ se usa:", COLD)

comp = ranked(pd.concat([direct, borrowed if COLD == "vecino visual" else popular]))
print(f"Anclas cubiertas: {comp.a.nunique():,} ({np.isin(emb_rows, comp.a).mean():.1%} de los artículos con imagen)")

results[f"Co-compra + arranque en frío ({COLD})"] = {"complementos": recall_at_k(comp, vp_comp), "general": np.nan}
res = pd.DataFrame(results).T.add_prefix(f"recall@{K} ")
res.to_csv(OUT / "complementary_eval.csv")
res.style.format("{:.4f}")

## Función `complements()`
Dada una o varias anclas, suma los scores de sus complementos y excluye los grupos de producto
que ya están en las anclas (si das una playera y un pantalón, propone zapatos, accesorios, etc.).

In [ ]:
comp_tbl = comp.assign(anchor_id=AID[comp.a], article_id=AID[comp.b])[
    ["anchor_id", "article_id", "score", "rank", "fuente"]].reset_index(drop=True)
meta = articles.set_index("article_id")[["prod_name", "product_type_name", "product_group_name",
                                         "colour_group_name", "index_group_name"]]

def complements(article_ids, k=12):
    """Prendas complementarias para una lista de article_id ancla → DataFrame ordenado por score."""
    article_ids = list(article_ids)
    c = comp_tbl[comp_tbl.anchor_id.isin(article_ids) & ~comp_tbl.article_id.isin(article_ids)]
    c = c.groupby("article_id", as_index=False).score.sum().join(meta, on="article_id")
    c = c[~c.product_group_name.isin(meta.loc[article_ids, "product_group_name"])]
    return c.nlargest(k, "score").reset_index(drop=True)

for aid in train.article_id.value_counts().index[[0, 50, 500]]:
    print("ANCLA:", aid, "|", " · ".join(meta.loc[aid, ["prod_name", "product_type_name", "colour_group_name", "index_group_name"]]))
    display(complements([aid], 6)[["article_id", "prod_name", "product_type_name", "colour_group_name", "score"]])

In [ ]:
t0 = time.time(); complements(train.article_id.iloc[:3]); print(f"Latencia complements(): {1000*(time.time()-t0):.0f} ms")

## Guardar

In [ ]:
cop = top_k(pairs, BEST, 50)
pd.DataFrame({"anchor_id": AID[cop.a], "article_id": AID[cop.b], "score": cop[BEST].to_numpy(),
              "count": cop["count"].to_numpy(), "rank": cop["rank"].to_numpy()}).to_parquet(OUT / "copurchase.parquet")
comp_tbl.to_parquet(OUT / "complements.parquet")
print(os.listdir(OUT))